# Week 5 Exercises

In [24]:
import numpy as np
import sys
sys.path.append('..')
from CV_functions import box3d, Pi, PiInv, projectpoints, CrossOp

R = np.eye(3)
t1 = np.array([[0], [0], [1]])
t2 = np.array([[0], [0], [20]])
K = np.array([
    [700, 0, 600], 
    [0, 700, 400], 
    [0, 0, 1]
])
Q = np.array([1, 1, 0]).reshape(3, 1)

### Ex 5.1

In [25]:
P1 = K @ np.hstack((R, t1))
P2 = K @ np.hstack((R, t2))

q1 = Pi(P1 @ PiInv(Q))
q2 = Pi(P2 @ PiInv(Q))

print(q1, q2)

[[1300.]
 [1100.]] [[635.]
 [435.]]


### Ex 5.2

In [31]:
def triangulate(qs, Ps):
    """
    qs: list of n pixel coords, each shape (2,) or (2,1) — inhomogeneous
    Ps: list of n projection matrices, each shape (3,4)
    Returns: Q, the triangulated 3D point (homogeneous, shape (4,1))
    """
    B = []
    for q, P in zip(qs, Ps):
        q = np.array(q).flatten()
        x, y = q[0], q[1]
        B.append(x * P[2] - P[0])
        B.append(y * P[2] - P[1])
    B = np.array(B)

    U, S, Vt = np.linalg.svd(B)
    Q = Vt[-1]          # smallest singular vector = null-space solution
    Q = Q / Q[-1]       # normalize so last coord is 1
    return Q.reshape(-1, 1)

q1_tilde = q1 + np.array([[1], [-1]])
q2_tilde = q2 + np.array([[1], [-1]])

Q_tilde = triangulate([q1_tilde, q2_tilde], [P1, P2])

q1_tilde_tri = Pi(P1 @ Q_tilde)
q2_tilde_tri = Pi(P2 @ Q_tilde)

print(np.linalg.norm(q1_tilde - q1_tilde_tri))
print(np.linalg.norm(q2_tilde - q2_tilde_tri))

print(np.linalg.norm(Pi(Q_tilde) - Q))

13.433018988192023
0.6717725840473774
0.02122181735338106


### Ex 5.3

In [32]:
from scipy.optimize import least_squares

def triangulate_nonlin(qs, Ps):
    """
    qs: list of n (2,) or (2,1) pixel coordinates, one per view.
    Ps: list of n (3,4) projection matrices, same order as qs.
    Returns: Q (3,1) the refined 3D point.
    """
    def compute_residuals(Q):
        Qh = np.append(Q, 1).reshape(4, 1)  # homogenize the 3-param Q
        residuals = []
        for P, q in zip(Ps, qs):
            q_proj = Pi(P @ Qh)               # (2,1)
            q_true = np.asarray(q).reshape(2, 1)
            residuals.append((q_proj - q_true).flatten())
        return np.concatenate(residuals)

    Q0_h = triangulate(qs, Ps)      # initial guess (linear algorithm)
    Q0 = Pi(Q0_h).flatten()         # (3,) non-homogeneous starting point

    result = least_squares(compute_residuals, Q0)
    Q_est = result.x.reshape(3, 1)
    return Q_est

### Ex 5.4

In [39]:
Q_hat = triangulate_nonlin([q1_tilde, q2_tilde], [P1, P2])

q1_hat = Pi(P1 @ PiInv(Q_hat))
q2_hat = Pi(P2 @ PiInv(Q_hat))

print(np.linalg.norm(q1_tilde - q1_hat))
print(np.linalg.norm(q2_tilde - q2_hat))

print(np.linalg.norm(Q_hat - Q))

0.06701026983062001
1.3401508668481283
0.002117415594345091
